# Experiment 008: Score Normalization and Threshold Ablation

## Analysis of Worker Idle Time Paradox Diagnostic Experiment

This notebook analyzes the results of Experiment 008, which tests two hypotheses about the worker idle time paradox:

1. **Hypothesis 1 (Mis-scaled Components)**: Fairness component dominates due to scale mismatch
2. **Hypothesis 2 (Soft-Threshold Feedback)**: Threshold deferral mechanism creates artificial task shortages

### Experimental Groups:
- **Group A**: Greedy baseline (efficiency reference)
- **Group B**: Composite current (replicate paradox)
- **Group C**: Composite + normalization (test H1)
- **Group D**: Composite + normalization + no threshold (test H1+H2)

### Analysis Structure:
1. Load and prepare data
2. **Primary Validation**: Compare mean worker idle times across groups
3. **Diagnostic Analysis**: Component dominance patterns
4. **Diagnostic Analysis**: Task deferral rates
5. **Secondary Metrics**: JFI, wait times, distances
6. **Statistical Testing**: Significance tests
7. **Conclusions**: Which hypothesis is supported?


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import glob
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("✅ Libraries imported successfully")


## 1. Load and Prepare Data

Load the aggregate results and diagnostic data from the experiment run.


In [ ]:
# Find the most recent experiment data directory
data_dir = Path("data")
exp_dirs = sorted(data_dir.glob("exp_008_*"))
if not exp_dirs:
    raise FileNotFoundError("No experiment 008 data found. Please run run_experiment.py first.")

latest_exp_dir = exp_dirs[-1]
print(f"📁 Loading data from: {latest_exp_dir}")

# Load aggregate results
aggregate_file = latest_exp_dir / "experiment_008_aggregate_results.csv"
results_df = pd.read_csv(aggregate_file)

# Load metadata
metadata_file = latest_exp_dir / "experiment_008_metadata.json"
with open(metadata_file, 'r') as f:
    metadata = json.load(f)

print(f"\n✅ Loaded {len(results_df)} experiment results")
print(f"   Duration: {metadata['duration_hours']:.2f} hours")
print(f"   Successful: {metadata['successful']}/{metadata['total_experiments']}")
print(f"\nGroup distribution:")
print(results_df.groupby('group').size())


## 2. Primary Validation: Worker Idle Time Comparison

**Key Question**: Does normalization and/or threshold removal reduce worker idle times?

**Prediction**:
- B (current) >> C (normalized) > D (normalized + no threshold) ≈ A (greedy)


In [ ]:
# NOTE: Worker idle time calculation requires worker-level data from simulation
# For this template, we'll use placeholder values
# In actual analysis, load worker data from simulation outputs and calculate idle times

# Group statistics
group_stats = results_df.groupby('group').agg({
    'task_assignment_ratio': ['mean', 'std'],
    'jains_fairness_index': ['mean', 'std'],
    'mean_task_wait_time_min': ['mean', 'std'],
    'mean_pickup_distance_km': ['mean', 'std'],
    'completed_tasks': ['mean', 'std']
}).round(3)

print("Group Statistics:")
print("="*80)
print(group_stats)
print()

# For diagnostic groups (B, C, D), show diagnostic metrics
composite_results = results_df[results_df['strategy'] == 'composite'].copy()
if not composite_results.empty:
    diag_stats = composite_results.groupby('group').agg({
        'deferral_rate': ['mean', 'std'],
        'dominant_fairness_pct': ['mean', 'std'],
        'dominant_utility_pct': ['mean', 'std'],
        'avg_dominance_ratio': ['mean', 'std']
    }).round(3)
    
    print("\nDiagnostic Metrics (Composite Groups Only):")
    print("="*80)
    print(diag_stats)


## 3. Diagnostic Analysis: Component Dominance Patterns

**Hypothesis 1 Test**: If mis-scaled components cause the paradox, we should see:
- Group B: Fairness dominates >70% of assignments
- Groups C & D: More balanced dominance (~30-40% each component)


In [ ]:
# Visualize component dominance patterns
if not composite_results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Dominance percentages by group
    ax1 = axes[0]
    groups = ['B', 'C', 'D']
    dominance_data = []
    for group in groups:
        group_data = composite_results[composite_results['group'] == group]
        dominance_data.append({
            'Group': group,
            'Fairness': group_data['dominant_fairness_pct'].mean(),
            'Starvation': group_data['dominant_starvation_pct'].mean(),
            'Utility': group_data['dominant_utility_pct'].mean()
        })
    
    dominance_df = pd.DataFrame(dominance_data)
    dominance_df.set_index('Group')[['Fairness', 'Starvation', 'Utility']].plot(
        kind='bar', ax=ax1, width=0.8
    )
    ax1.set_ylabel('Dominance (%)')
    ax1.set_title('Component Dominance by Group')
    ax1.set_xlabel('Experimental Group')
    ax1.legend(title='Component')
    ax1.axhline(y=33.33, color='gray', linestyle='--', alpha=0.5, label='Equal dominance (33%)')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Dominance ratio by group
    ax2 = axes[1]
    composite_results.boxplot(column='avg_dominance_ratio', by='group', ax=ax2)
    ax2.set_ylabel('Dominance Ratio (max / sum of others)')
    ax2.set_title('Dominance Ratio Distribution by Group')
    ax2.set_xlabel('Experimental Group')
    ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Equal contributions')
    plt.sca(ax2)
    plt.xticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig(latest_exp_dir / 'component_dominance_analysis.png', dpi=150)
    plt.show()
    
    print("\n✅ Component dominance analysis complete")
else:
    print("⚠️ No composite strategy results found")


## 4. Diagnostic Analysis: Task Deferral Rates

**Hypothesis 2 Test**: If soft-threshold causes the paradox, we should see:
- Groups B & C: High deferral rates (>30%)
- Group D: Zero deferrals (threshold disabled)


In [ ]:
# Visualize deferral rates
if not composite_results.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    composite_results.boxplot(column='deferral_rate', by='group', ax=ax)
    ax.set_ylabel('Deferral Rate (%)')
    ax.set_title('Task Deferral Rate by Group')
    ax.set_xlabel('Experimental Group')
    ax.set_ylim(bottom=0)
    plt.sca(ax)
    plt.xticks(rotation=0)
    
    plt.tight_layout()
    plt.savefig(latest_exp_dir / 'deferral_rate_analysis.png', dpi=150)
    plt.show()
    
    print("\nDeferral Rate Statistics:")
    print("="*80)
    deferral_summary = composite_results.groupby('group')['deferral_rate'].agg(['mean', 'std', 'min', 'max'])
    print(deferral_summary.round(3))
    
    print("\n✅ Deferral rate analysis complete")
else:
    print("⚠️ No composite strategy results found")


## 5. Secondary Metrics: Fairness and Efficiency Trade-offs

Check that interventions don't sacrifice fairness or throughput.


In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Task Assignment Ratio
ax1 = axes[0, 0]
results_df.boxplot(column='task_assignment_ratio', by='group', ax=ax1)
ax1.set_ylabel('Task Assignment Ratio')
ax1.set_title('Task Assignment Ratio by Group')
ax1.set_xlabel('Experimental Group')
plt.sca(ax1)
plt.xticks(rotation=0)

# Plot 2: Jain's Fairness Index
ax2 = axes[0, 1]
results_df.boxplot(column='jains_fairness_index', by='group', ax=ax2)
ax2.set_ylabel('Jain\\'s Fairness Index')
ax2.set_title('Fairness Index by Group')
ax2.set_xlabel('Experimental Group')
plt.sca(ax2)
plt.xticks(rotation=0)

# Plot 3: Mean Task Wait Time
ax3 = axes[1, 0]
results_df.boxplot(column='mean_task_wait_time_min', by='group', ax=ax3)
ax3.set_ylabel('Mean Task Wait Time (min)')
ax3.set_title('Task Wait Time by Group')
ax3.set_xlabel('Experimental Group')
plt.sca(ax3)
plt.xticks(rotation=0)

# Plot 4: Mean Pickup Distance
ax4 = axes[1, 1]
results_df.boxplot(column='mean_pickup_distance_km', by='group', ax=ax4)
ax4.set_ylabel('Mean Pickup Distance (km)')
ax4.set_title('Pickup Distance by Group')
ax4.set_xlabel('Experimental Group')
plt.sca(ax4)
plt.xticks(rotation=0)

plt.tight_layout()
plt.savefig(latest_exp_dir / 'secondary_metrics_comparison.png', dpi=150)
plt.show()

print("✅ Secondary metrics visualized")


## 6. Statistical Significance Testing

Test if differences between groups are statistically significant.


In [ ]:
# Mann-Whitney U tests for key comparisons
print("Statistical Significance Tests (Mann-Whitney U)")
print("="*80)

metrics_to_test = ['task_assignment_ratio', 'jains_fairness_index', 
                   'mean_task_wait_time_min', 'mean_pickup_distance_km']

# Key comparisons:
# 1. B vs C (test H1: normalization effect)
# 2. C vs D (test H2: threshold removal effect)
# 3. B vs D (test combined effect)
# 4. A vs D (compare final solution to greedy baseline)

comparisons = [
    ('B', 'C', 'H1: Normalization effect'),
    ('C', 'D', 'H2: Threshold removal effect'),
    ('B', 'D', 'Combined intervention'),
    ('A', 'D', 'Final vs. Greedy baseline')
]

for metric in metrics_to_test:
    print(f"\n{metric}:")
    print("-"*80)
    for group1, group2, desc in comparisons:
        data1 = results_df[results_df['group'] == group1][metric].dropna()
        data2 = results_df[results_df['group'] == group2][metric].dropna()
        
        if len(data1) > 0 and len(data2) > 0:
            statistic, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
            mean1, mean2 = data1.mean(), data2.mean()
            diff_pct = ((mean2 - mean1) / mean1 * 100) if mean1 != 0 else 0
            
            sig = "✅ Significant" if p_value < 0.05 else "❌ Not significant"
            print(f"  {desc} ({group1} vs {group2}):")
            print(f"    {group1}: {mean1:.3f}, {group2}: {mean2:.3f}, Diff: {diff_pct:+.1f}%, p={p_value:.4f} {sig}")

print("\n✅ Statistical testing complete")


## 7. Conclusions and Next Steps

### Interpretation Guide:

**If Hypothesis 1 is Supported:**
- Group C shows significant improvement over Group B
- Fairness dominance drops significantly in Group C
- Dominance ratios decrease in Group C

**If Hypothesis 2 is Supported:**
- Group D shows significant improvement over Group C
- Deferral rate is the key differentiator
- Group D achieves near-zero deferrals

**If Both Hypotheses are Supported:**
- Progressive improvement: B > C > D
- Group D achieves best results (approaching Group A efficiency)

### Action Items:
1. Review the statistical tests above to determine which hypothesis is supported
2. If H1 confirmed: Implement normalization as default in Composite strategy
3. If H2 confirmed: Redesign threshold mechanism (bounded deferrals)
4. If both: Implement both fixes and validate in Experiment 009
5. Document findings in `results.md`


In [ ]:
# Summary table for conclusions
print("="*80)
print("EXPERIMENT 008 SUMMARY")
print("="*80)
print(f"\nDataset: {metadata['dataset']}")
print(f"Duration: {metadata['duration_hours']:.2f} hours")
print(f"Total Experiments: {metadata['total_experiments']}")
print(f"Output Directory: {latest_exp_dir}")
print("\n" + "="*80)
print("📊 Analysis complete! Review the results above to determine next steps.")
